In [ ]:
# AI experiment pipeline notebook (single-script style)
# ----------------------------------------------------
# This script:
# 1) Loads prompt data from CSV
# 2) Loads baseline/context templates
# 3) Calls GPT and Claude for each row
# 4) Saves outputs to data/results.csv

from pathlib import Path
import time
import pandas as pd

from openai import OpenAI
from anthropic import Anthropic


# Step 1: Set up project paths
# Using notebook location so paths work when run from notebooks/experiment.ipynb
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "prompts.csv"
BASELINE_TEMPLATE_PATH = PROJECT_ROOT / "prompts" / "baseline.txt"
CONTEXT_TEMPLATE_PATH = PROJECT_ROOT / "prompts" / "context.txt"
RESULTS_PATH = PROJECT_ROOT / "data" / "results.csv"


# Step 2: Load input data and templates
prompts_df = pd.read_csv(DATA_PATH)

with open(BASELINE_TEMPLATE_PATH, "r", encoding="utf-8") as f:
    baseline_template = f.read()

with open(CONTEXT_TEMPLATE_PATH, "r", encoding="utf-8") as f:
    context_template = f.read()


# Step 3: Normalize placeholders and format prompts
# Your instruction uses {candidate_info}, while files currently use {candidate_information}.
# We normalize both templates to keep one simple formatting function.
baseline_template = baseline_template.replace("{candidate_information}", "{candidate_info}")
context_template = context_template.replace("{candidate_information}", "{candidate_info}")


def format_prompt(template: str, candidate_info: str, job_description: str) -> str:
    """Fill one prompt template with row values."""
    return template.format(
        candidate_info=candidate_info,
        job_description=job_description,
    )


# Step 4: API clients (lazy init)
# Make sure environment variables are set before running:
# OPENAI_API_KEY and ANTHROPIC_API_KEY
openai_client = None
anthropic_client = None


# Step 5: Model call helpers with basic error handling + one retry
# If a call fails, retry once after a short delay.

def call_gpt(prompt: str) -> str:
    """Call OpenAI GPT model and return text output."""
    global openai_client

    if openai_client is None:
        try:
            openai_client = OpenAI()
        except Exception as e:
            return f"ERROR (GPT): {e}"

    for attempt in range(2):
        try:
            response = openai_client.responses.create(
                model="gpt-4o-mini",
                input=prompt,
            )
            return response.output_text
        except Exception as e:
            if attempt == 0:
                time.sleep(1)
            else:
                return f"ERROR (GPT): {e}"



def call_claude(prompt: str) -> str:
    """Call Anthropic Claude model and return text output."""
    global anthropic_client

    if anthropic_client is None:
        try:
            anthropic_client = Anthropic()
        except Exception as e:
            return f"ERROR (Claude): {e}"

    for attempt in range(2):
        try:
            response = anthropic_client.messages.create(
                model="claude-3-haiku-20240307",
                max_tokens=1200,
                messages=[
                    {"role": "user", "content": prompt}
                ],
            )

            text_parts = []
            for block in response.content:
                if block.type == "text":
                    text_parts.append(block.text)

            return "\n".join(text_parts).strip()
        except Exception as e:
            if attempt == 0:
                time.sleep(1)
            else:
                return f"ERROR (Claude): {e}"


# Step 6: Test on ONE row first
first_row = prompts_df.iloc[0]

# Support either candidate_information (current CSV) or candidate_info (future CSV)
candidate_value = first_row.get("candidate_information", first_row.get("candidate_info", ""))
job_value = first_row["job_description"]

first_baseline_prompt = format_prompt(baseline_template, candidate_value, job_value)
first_context_prompt = format_prompt(context_template, candidate_value, job_value)

print("=== ONE-ROW TEST: GPT baseline ===")
print(call_gpt(first_baseline_prompt))
print("\n=== ONE-ROW TEST: GPT context ===")
print(call_gpt(first_context_prompt))
print("\n=== ONE-ROW TEST: Claude baseline ===")
print(call_claude(first_baseline_prompt))
print("\n=== ONE-ROW TEST: Claude context ===")
print(call_claude(first_context_prompt))


# Step 7: Loop through all rows and collect outputs
results = []

total = len(prompts_df)
for i, row in prompts_df.iterrows():
    print(f"Processing prompt {i + 1}/{total}")

    candidate_value = row.get("candidate_information", row.get("candidate_info", ""))
    job_value = row["job_description"]

    baseline_prompt = format_prompt(baseline_template, candidate_value, job_value)
    context_prompt = format_prompt(context_template, candidate_value, job_value)

    # GPT baseline
    gpt_baseline_output = call_gpt(baseline_prompt)
    results.append(
        {
            "prompt_id": row["prompt_id"],
            "model": "gpt",
            "prompt_type": "baseline",
            "output": gpt_baseline_output,
            "output_length": len(gpt_baseline_output),
        }
    )

    # GPT context
    gpt_context_output = call_gpt(context_prompt)
    results.append(
        {
            "prompt_id": row["prompt_id"],
            "model": "gpt",
            "prompt_type": "context",
            "output": gpt_context_output,
            "output_length": len(gpt_context_output),
        }
    )

    # Claude baseline
    claude_baseline_output = call_claude(baseline_prompt)
    results.append(
        {
            "prompt_id": row["prompt_id"],
            "model": "claude",
            "prompt_type": "baseline",
            "output": claude_baseline_output,
            "output_length": len(claude_baseline_output),
        }
    )

    # Claude context
    claude_context_output = call_claude(context_prompt)
    results.append(
        {
            "prompt_id": row["prompt_id"],
            "model": "claude",
            "prompt_type": "context",
            "output": claude_context_output,
            "output_length": len(claude_context_output),
        }
    )


# Step 8: Save results to CSV
results_df = pd.DataFrame(results)
results_df.to_csv(RESULTS_PATH, index=False)

print(f"\nDone. Saved {len(results_df)} rows to: {RESULTS_PATH}")
print(results_df.head())


# Step 9: Quick sanity checks + small spot-check sample
print("\n=== Sanity checks ===")
print("Empty outputs:", results_df["output"].fillna("").str.strip().eq("").sum())
print("Total rows:", len(results_df))

expected_combos = {("gpt", "baseline"), ("gpt", "context"), ("claude", "baseline"), ("claude", "context")}
missing_combo_count = 0
for prompt_id, group in results_df.groupby("prompt_id"):
    combos = set(zip(group["model"], group["prompt_type"]))
    if expected_combos - combos:
        missing_combo_count += 1
print("Prompts with missing model/prompt combinations:", missing_combo_count)


# Small random sample for manual quality review (3 prompts)
print("\n=== Spot-check sample (3 random prompt_ids) ===")
sample_ids = prompts_df["prompt_id"].sample(3, random_state=42).tolist()
for prompt_id in sample_ids:
    print(f"\n--- {prompt_id} ---")
    rows = results_df[results_df["prompt_id"] == prompt_id]
    for _, r in rows.iterrows():
        preview = str(r["output"]).replace("\n", " ")[:220]
        print(f"{r['model']}/{r['prompt_type']} | len={r['output_length']} | {preview}")
